In [2]:
n = 19
g = 1.2
mu = 2.5
T = 50

In [5]:
import numpy as np
from scipy.optimize import root

def model(X, n, g, mu):
    """
    Defines the system of ODEs (without time t as it's not needed for equilibrium)

    Input:
    - X: array of variables [x_0, x_1, ..., x_{n-1}]
    - g, mu: scalar model parameters
    - n: number of ODEs (size of the system)

    Returns:
    - dXdt: array of derivatives [dx_0/dt, dx_1/dt, ..., dx_{n-1}/dt]
    """
    dXdt = np.zeros(n)
    dXdt[0] = X[0] * (1 - g * X[-2] - X[0] - g * X[1] + mu * X[-3])
    dXdt[1] = X[1] * (1 - g * X[-1] - X[1] - g * X[2])
    dXdt[2:n-1] = X[2:n-1] * (1 - g * X[0:n-3] - X[2:n-1] - g * X[3:n])
    dXdt[-1] = X[-1] * (1 - g * X[-3] - X[-1] - g * X[0])
    return dXdt

eventus = root(model, np.ones(n),  method='krylov', args=(n,g,mu))
x_bar = eventus.x

B= np.zeros([n,n])
for i in range(2,n-1):
    B[i,i] = -1
    B[i,i-2] = -g
    B[i,i+1] = -g

B[0,0] = -1
B[0,n-2] = -g
B[0,1] = -g
B[0,n-3] = mu

B[1,1] = -1
B[1,n-1] = -g
B[1,2] = -g

B[n-1,n-1] = -1
B[n-1,n-3] = -g
B[n-1,0] = -g

M = np.zeros([n,n])

for i in range(n):
      M[i,i] = 1 + np.matmul(B[i,:], x_bar)
for i in range(n):
    for j in range(n):
                M[i,j] += B[i, j]*x_bar[i]

eigenvalores, eigenvectores = np.linalg.eig(M)
max_eigenvalue_index = np.argmax(np.real(eigenvalores))
    
x_0 = eigenvectores[:, max_eigenvalue_index]

max_ratio = np.real(eigenvalores[max_eigenvalue_index])

In [ ]:
import numpy as np
from scipy.optimize import root
import scipy

def part1q1a(n, g, mu, T):
    """Part 1, question 1 (a)
    Use the variable inputs if/as needed.
    Input:
    n: number of ODEs
    g,mu: model parameters
    T: time at which perturbation energy ratio should be maximized

    Output:
    xbar: n-element array containing non-trivial equilibrium solution
    xtilde0: n-element array corresponding to computed initial condition
    eratio: computed maximum perturbation energy ratio
    """
    xbar = np.zeros(n)
    xtilde0 = np.zeros(n)
    eratio = 0.0

    def model(X, n, g, mu):
        dXdt = np.zeros(n)
        dXdt[0] = X[0] * (1 - g * X[-2] - X[0] - g * X[1] + mu * X[-3])
        dXdt[1] = X[1] * (1 - g * X[-1] - X[1] - g * X[2])
        dXdt[2 : n - 1] = X[2 : n - 1] * (
            1 - g * X[0 : n - 3] - X[2 : n - 1] - g * X[3:n]
        )
        dXdt[-1] = X[-1] * (1 - g * X[-3] - X[-1] - g * X[0])
        return dXdt

    eventus = root(
        model,
        1 / (2 * g + 1) + 0.1 * np.random.rand(n),
        method="krylov",
        args=(n, g, mu),
    )
    x_bar = eventus.x

    B = np.zeros([n, n])
    for i in range(2, n - 1):
        B[i, i] = -1
        B[i, i - 2] = -g
        B[i, i + 1] = -g

    B[0, 0] = -1
    B[0, n - 2] = -g
    B[0, 1] = -g
    B[0, n - 3] = mu

    B[1, 1] = -1
    B[1, n - 1] = -g
    B[1, 2] = -g

    B[n - 1, n - 1] = -1
    B[n - 1, n - 3] = -g
    B[n - 1, 0] = -g

    M = np.zeros([n, n])

    for i in range(n):
        M[i, i] = 1 + np.matmul(B[i, :], x_bar)

    for i in range(n):
        for j in range(n):
            M[i, j] += B[i, j] * x_bar[i]

    matM = scipy.linalg.expm(M*T)

    U,Sigma,VT = np.linalg.svd(matM)

    eratio = Sigma[0]**2 

    xtilde0 = VT[0,:]

    xbar = x_bar

    return xbar, xtilde0, eratio